### FNN trained on protein scaling factor data

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import roc_curve
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import json
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Input

In [2]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# Function to load data
def load_data(matrix_folder, label_file):
    # Load labels
    with open(label_file, 'r') as f:
        labels = np.array([int(line.strip()) for line in f])

    # Load matrices
    matrices = []
    for file in sorted(os.listdir(matrix_folder)):
        if file.endswith('.csv'):
            matrix_path = os.path.join(matrix_folder, file)
            matrix = pd.read_csv(matrix_path, header=None, skiprows=1).values
            matrices.append(matrix)

    matrices = np.array(matrices)
    return matrices, labels

In [ ]:
# Function for 10-fold cross-validation training
def cross_validate_model(X, y, n_splits=10):
    X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_trainval = scaler.fit_transform(X_trainval)
    X_test = scaler.transform(X_test)

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_accuracies = []
    fold_roc_aucs = []
    fold_reports = []
    fold_confusion_matrices = []

    fold_num = 1
    for train_index, val_index in kfold.split(X_trainval):
        print(f"\nTraining Fold {fold_num}/{n_splits}...")
        X_train, X_val = X_trainval[train_index], X_trainval[val_index]
        y_train, y_val = y_trainval[train_index], y_trainval[val_index]

        model = Sequential([
            Input(shape=(X_train.shape[1],)),
            Dense(64, activation='sigmoid', kernel_regularizer=l2(0.001)),
            Dense(32, activation='sigmoid', kernel_regularizer=l2(0.001)),
            Dropout(0.4),
            Dense(16, activation='sigmoid', kernel_regularizer=l2(0.001)),
            Dropout(0.4),
            Dense(1, activation='sigmoid')  # Binary classification
        ]) 

        model.compile(optimizer=Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])


        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=32,
            verbose=1
        )

        # Predict probabilities and labels for the validation set
        y_test_probs = model.predict(X_test, verbose=0)
        y_test_pred = (y_test_probs > 0.5).astype(int)


        # Calculate metrics for the fold
        accuracy = accuracy_score(y_test, y_test_pred)
        roc_auc = roc_auc_score(y_test, y_test_probs)
        classification_rep = classification_report(y_test, y_test_pred, target_names=['Negative', 'Positive'], output_dict=True)
        confusion_mat = confusion_matrix(y_test, y_test_pred)

        # Append results for the fold
        fold_accuracies.append(accuracy)
        fold_roc_aucs.append(roc_auc)
        fold_reports.append(classification_rep)
        fold_confusion_matrices.append(confusion_mat)

        # Calculate average metrics across all folds
        avg_accuracy = np.mean(fold_accuracies)
        avg_auc = np.mean(fold_roc_aucs)

        print(f"Fold {fold_num} Accuracy: {accuracy:.4f}, ROC-AUC: {roc_auc:.4f}")
        fold_num += 1
        
    avg_classification_report = {
        'Positive': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Positive']['precision'] for r in fold_reports])),
            'recall': float(np.mean([r['Positive']['recall'] for r in fold_reports])),
            'f1-score': float(np.mean([r['Positive']['f1-score'] for r in fold_reports]))
        },
        'Negative': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Negative']['precision'] for r in fold_reports])),
            'recall': float(np.mean([r['Negative']['recall'] for r in fold_reports])),
            'f1-score': float(np.mean([r['Negative']['f1-score'] for r in fold_reports]))
        }
    }

    print("\nAverage Classification Report (across all folds):")
    print(avg_classification_report)


    return fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report


Saving the results into a JSON

In [5]:
def save_classification_report(dataset_name, avg_classification_report):
    results_file = "reports/nosf/classification_reports2_ffnn.json"

    # Add dataset name to the report
    report_to_save = {
        "dataset": dataset_name,
        "report": avg_classification_report
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(report_to_save)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Classification report saved to {results_file}")

In [6]:
def save_results(dataset_name, fold_accuracies, fold_roc_aucs, fold_reports):
    results_file = "reports/nosf/classification_results2_FNN.json"

    # Extract precision, recall, and f1-score for each fold
    fold_metrics = []
    for report in fold_reports:
        fold_metrics.append({
            "Negative": {
                "precision": report["Negative"]["precision"],
                "recall": report["Negative"]["recall"],
                "f1-score": report["Negative"]["f1-score"]
            },
            "Positive": {
                "precision": report["Positive"]["precision"],
                "recall": report["Positive"]["recall"],
                "f1-score": report["Positive"]["f1-score"]
            }
        })

    # Convert results to a dictionary
    results_dict = {
        "dataset": dataset_name,
        "accuracies": fold_accuracies,
        "roc_aucs": fold_roc_aucs,
        "fold_metrics": fold_metrics
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(results_dict)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Results saved to {results_file}")



RES 25

In [7]:
# Load dataset
matrix_folder_antiinflam = 'data/matrices/mat_nosf/mat25/aip_antiinflam_matrix'  
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.5823 - loss: 0.8874 - val_accuracy: 0.5941 - val_loss: 0.7652
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5789 - loss: 0.7694 - val_accuracy: 0.5941 - val_loss: 0.7341
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5503 - loss: 0.7571 - val_accuracy: 0.5941 - val_loss: 0.7203
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5540 - loss: 0.7401 - val_accuracy: 0.5941 - val_loss: 0.7116
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5681 - loss: 0.7174 - val_accuracy: 0.5941 - val_loss: 0.7046
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5875 - loss: 0.7115 - val_accuracy: 0.5941 - val_loss: 0.6987
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5971 - loss: 0.7019 - val_accuracy: 0.5941 - val_loss: 0.6934
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5866 - loss: 0.7021 - val_accu

In [8]:
save_results("aip_antiinflam_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [9]:
save_classification_report("aip_antiinflam_25", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [10]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_nosf/mat25/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.4878 - loss: 1.0058 - val_accuracy: 0.5072 - val_loss: 0.8526
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5272 - loss: 0.8642 - val_accuracy: 0.5072 - val_loss: 0.7902
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5702 - loss: 0.7920 - val_accuracy: 0.6522 - val_loss: 0.7645
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5641 - loss: 0.7685 - val_accuracy: 0.6377 - val_loss: 0.7462
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.5204 - loss: 0.7754 - val_accuracy: 0.6957 - val_loss: 0.7291
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5693 - loss: 0.7742 - val_accuracy: 0.7246 - val_loss: 0.7112
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5672 - loss: 0.7515 - val_accuracy: 0.8116 - val_loss: 0.6896
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6177 - loss: 0.7045 - v

In [11]:
save_results("amp_antibp_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [12]:
save_classification_report("amp_antibp_25", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [13]:
# Load dataset
matrix_folder_antipb2 = 'data/matrices/mat_nosf/mat25/amp_antibp2_matrix' 
label_file_antipb2 = 'data/labels/amp_antibp2.txt'  
matrices, labels = load_data(matrix_folder_antipb2, label_file_antipb2)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.4966 - loss: 0.9561 - val_accuracy: 0.4625 - val_loss: 0.7889
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5281 - loss: 0.7973 - val_accuracy: 0.7625 - val_loss: 0.7464
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5695 - loss: 0.7744 - val_accuracy: 0.7750 - val_loss: 0.7174
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5753 - loss: 0.7347 - val_accuracy: 0.8250 - val_loss: 0.6805
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6736 - loss: 0.6802 - val_accuracy: 0.8188 - val_loss: 0.6412
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7268 - loss: 0.6508 - val_accuracy: 0.8125 - val_loss: 0.6051
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7201 - loss: 0.6283 - val_accuracy: 0.8062 - val_loss: 0.5832
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7408 - loss: 0.6217 - val_accu

In [14]:
save_results("amp_antibp2_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [15]:
save_classification_report("amp_antibp2_25", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [16]:
# Load dataset
matrix_folder_csamp = 'data/matrices/mat_nosf/mat25/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 83ms/step - accuracy: 0.4863 - loss: 0.9380 - val_accuracy: 0.5714 - val_loss: 0.8464
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4519 - loss: 0.9383 - val_accuracy: 0.5714 - val_loss: 0.8308
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5000 - loss: 0.8721 - val_accuracy: 0.5714 - val_loss: 0.8187
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4775 - loss: 0.8558 - val_accuracy: 0.5714 - val_loss: 0.8085
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5520 - loss: 0.8034 - val_accuracy: 0.5714 - val_loss: 0.7992
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4922 - loss: 0.8402 - val_accuracy: 0.5714 - val_loss: 0.7914
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5482 - loss: 0.8034 - val_accuracy: 0.5714 - val_loss: 0.7850
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5084 - loss: 0.7939 - val_accuracy: 0.6

c:\Users\User\anaconda3\envs\test_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\User\anaconda3\envs\test_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\User\anaconda3\envs\test_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 146ms/step - accuracy: 0.4987 - loss: 1.0743 - val_accuracy: 0.6500 - val_loss: 0.8454
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.4848 - loss: 1.0535 - val_accuracy: 0.6500 - val_loss: 0.8137
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5169 - loss: 1.0022 - val_accuracy: 0.6500 - val_loss: 0.7913
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4948 - loss: 0.9190 - val_accuracy: 0.6500 - val_loss: 0.7775
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4878 - loss: 0.8827 - val_accuracy: 0.6500 - val_loss: 0.7703
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4771 - loss: 0.8667 - val_accuracy: 0.6500 - val_loss: 0.7672
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5148 - loss: 0.8211 - val_accuracy: 0.6500 - val_loss: 0.7674
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6025 - loss: 0.7869 - val_accuracy: 0.6500 - val_loss: 0.7689
Epoch 9/15

c:\Users\User\anaconda3\envs\test_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\User\anaconda3\envs\test_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\User\anaconda3\envs\test_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
save_results("amp_csamp_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [18]:
save_classification_report("amp_csamp_25", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [19]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/mat_nosf/mat25/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.4841 - loss: 1.1122 - val_accuracy: 0.5200 - val_loss: 0.8910
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5035 - loss: 0.9412 - val_accuracy: 0.5200 - val_loss: 0.8201
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5371 - loss: 0.8536 - val_accuracy: 0.5200 - val_loss: 0.7921
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5033 - loss: 0.8567 - val_accuracy: 0.6200 - val_loss: 0.7793
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5691 - loss: 0.7865 - val_accuracy: 0.4600 - val_loss: 0.7714
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5343 - loss: 0.8007 - val_accuracy: 0.5000 - val_loss: 0.7653
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5353 - loss: 0.7975 - val_accuracy: 0.5200 - val_loss: 0.7583
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4650 - loss: 0.8076 - 

In [20]:
save_results("hiv_ddi_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [21]:
save_classification_report("hiv_ddi_25", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [22]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/mat_nosf/mat25/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - accuracy: 0.5194 - loss: 0.9013 - val_accuracy: 0.5000 - val_loss: 0.8424
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5056 - loss: 0.8773 - val_accuracy: 0.5000 - val_loss: 0.8150
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4889 - loss: 0.8597 - val_accuracy: 0.5000 - val_loss: 0.7947
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5300 - loss: 0.8094 - val_accuracy: 0.5000 - val_loss: 0.7783
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5076 - loss: 0.7918 - val_accuracy: 0.5000 - val_loss: 0.7654
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5520 - loss: 0.7671 - val_accuracy: 0.5000 - val_loss: 0.7565
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5238 - loss: 0.7865 - val_accuracy: 0.5000 - val_loss: 0.7475
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5724 - loss: 0.7647 - v

In [23]:
save_results("hiv_lpv_25", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [24]:
save_classification_report("hiv_lpv_25", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [7]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_nosf/mat25/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.4703 - loss: 0.9542 - val_accuracy: 0.4237 - val_loss: 0.8555
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5218 - loss: 0.8626 - val_accuracy: 0.7288 - val_loss: 0.7889
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4999 - loss: 0.8259 - val_accuracy: 0.8136 - val_loss: 0.7569
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5796 - loss: 0.7638 - val_accuracy: 0.7966 - val_loss: 0.7340
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5577 - loss: 0.7661 - val_accuracy: 0.7627 - val_loss: 0.7128
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5226 - loss: 0.7835 - val_accuracy: 0.7797 - val_loss: 0.6904
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6500 - loss: 0.7021 - val_accuracy: 0.7797 - val_loss: 0.6645
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6714 - loss: 0.6669 - v

In [8]:
save_results("hiv_rtv_25", fold_accuracies, fold_roc_aucs, fold_reports)
save_classification_report("hiv_rtv_25", avg_classification_report)

Results saved to reports/nosf/classification_results2_FNN.json
Classification report saved to reports/nosf/classification_reports2_ffnn.json


RES 50

In [25]:
matrix_folder_antiinflam = 'data/matrices/aip_antiinflam_matrix'  
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)

# Flatten matrices if necessary
X = matrices.reshape(matrices.shape[0], -1)
y = labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)


Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.4759 - loss: 0.9209 - val_accuracy: 0.5941 - val_loss: 0.7562
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5577 - loss: 0.7837 - val_accuracy: 0.5941 - val_loss: 0.7394
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5617 - loss: 0.7737 - val_accuracy: 0.5941 - val_loss: 0.7277
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5485 - loss: 0.7507 - val_accuracy: 0.5941 - val_loss: 0.7182
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6065 - loss: 0.7166 - val_accuracy: 0.6000 - val_loss: 0.7062
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.6127 - loss: 0.7084 - val_accuracy: 0.6294 - val_loss: 0.6969
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6710 - loss: 0.6719 - val_accuracy: 0.6353 - val_loss: 0.6867
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7159 - loss: 0.6337 - v

In [26]:
save_results("aip_antiinflam_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [27]:
save_classification_report("aip_antiinflam_50", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [28]:

# Load dataset
matrix_folder_antipb = 'data/matrices/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)# Preprocess data
# Flatten matrices if necessary
X = matrices.reshape(matrices.shape[0], -1)
y = labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.4817 - loss: 0.8781 - val_accuracy: 0.6812 - val_loss: 0.7896
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6031 - loss: 0.7737 - val_accuracy: 0.6522 - val_loss: 0.7555
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5501 - loss: 0.7623 - val_accuracy: 0.8261 - val_loss: 0.7377
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5758 - loss: 0.7429 - val_accuracy: 0.8551 - val_loss: 0.7204
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5894 - loss: 0.7308 - val_accuracy: 0.8261 - val_loss: 0.6997
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6828 - loss: 0.6978 - val_accuracy: 0.8261 - val_loss: 0.6725
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6749 - loss: 0.6769 - val_accuracy: 0.8406 - val_loss: 0.6444
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7629 - loss: 0.6238 - v

In [29]:
save_results("amp_antibp_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [30]:
save_classification_report("amp_antibp_50", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [31]:
# Load dataset
matrix_folder_antipb2 = 'data/matrices/amp_antibp2_matrix' 
label_file_antipb2 = 'data/labels/amp_antibp2.txt'  
matrices, labels = load_data(matrix_folder_antipb2, label_file_antipb2)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels


# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.4975 - loss: 0.8781 - val_accuracy: 0.6750 - val_loss: 0.7628
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5369 - loss: 0.7823 - val_accuracy: 0.8125 - val_loss: 0.7263
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5506 - loss: 0.7544 - val_accuracy: 0.8250 - val_loss: 0.6764
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6861 - loss: 0.6836 - val_accuracy: 0.8125 - val_loss: 0.6245
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7296 - loss: 0.6266 - val_accuracy: 0.8250 - val_loss: 0.5885
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7976 - loss: 0.5671 - val_accuracy: 0.8125 - val_loss: 0.5815
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8451 - loss: 0.5265 - val_accuracy: 0.7875 - val_loss: 0.5870
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8639 - loss: 0.4853 - v

In [32]:
save_results("amp_antibp2_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [33]:
save_classification_report("amp_antibp2_50", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [34]:
# Load dataset
matrix_folder_csamp = 'data/matrices/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 125ms/step - accuracy: 0.4867 - loss: 1.2095 - val_accuracy: 0.5714 - val_loss: 0.9836
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4761 - loss: 1.0460 - val_accuracy: 0.5714 - val_loss: 0.9154
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4677 - loss: 1.1228 - val_accuracy: 0.5714 - val_loss: 0.8624
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4685 - loss: 1.0054 - val_accuracy: 0.5714 - val_loss: 0.8233
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5293 - loss: 0.8633 - val_accuracy: 0.5714 - val_loss: 0.7971
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4687 - loss: 0.8770 - val_accuracy: 0.5714 - val_loss: 0.7813
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5125 - loss: 0.8732 - val_accuracy: 0.5714 - val_loss: 0.7718
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4461 - loss: 0.8454 - val_accuracy: 0.

In [35]:
save_results("amp_csamp_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [36]:
save_classification_report("amp_csamp_50", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [37]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)

In [38]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.5425 - loss: 0.8723 - val_accuracy: 0.5800 - val_loss: 0.8080
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5179 - loss: 0.8391 - val_accuracy: 0.5200 - val_loss: 0.7757
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5443 - loss: 0.7938 - val_accuracy: 0.5000 - val_loss: 0.7614
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5878 - loss: 0.7503 - val_accuracy: 0.5600 - val_loss: 0.7521
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6012 - loss: 0.7311 - val_accuracy: 0.6000 - val_loss: 0.7433
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6222 - loss: 0.7301 - val_accuracy: 0.6000 - val_loss: 0.7350
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6412 - loss: 0.7263 - val_accuracy: 0.6000 - val_loss: 0.7299
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5817 - loss: 0.7174 - v

In [39]:
save_results("hiv_ddi_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [40]:
save_classification_report("hiv_ddi_50", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [41]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.4712 - loss: 1.1391 - val_accuracy: 0.5000 - val_loss: 0.9222
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5031 - loss: 0.9196 - val_accuracy: 0.5000 - val_loss: 0.8116
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5485 - loss: 0.8324 - val_accuracy: 0.5000 - val_loss: 0.7658
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5512 - loss: 0.7811 - val_accuracy: 0.7500 - val_loss: 0.7505
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5373 - loss: 0.7951 - val_accuracy: 0.6250 - val_loss: 0.7442
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5943 - loss: 0.7397 - val_accuracy: 0.5750 - val_loss: 0.7402
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5755 - loss: 0.7647 - val_accuracy: 0.5750 - val_loss: 0.7349
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5873 - loss: 0.7480 - v

In [42]:
save_results("hiv_lpv_50", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [43]:
save_classification_report("hiv_lpv_50", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [9]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_nosf/mat50/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.4898 - loss: 0.9163 - val_accuracy: 0.5763 - val_loss: 0.7757
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5193 - loss: 0.8137 - val_accuracy: 0.5932 - val_loss: 0.7355
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5608 - loss: 0.7458 - val_accuracy: 0.7627 - val_loss: 0.7105
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.6194 - loss: 0.7149 - val_accuracy: 0.7627 - val_loss: 0.6847
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6931 - loss: 0.6773 - val_accuracy: 0.7627 - val_loss: 0.6538
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6842 - loss: 0.6600 - val_accuracy: 0.7797 - val_loss: 0.6235
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7416 - loss: 0.6100 - val_accuracy: 0.7797 - val_loss: 0.5973
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7900 - loss: 0.5858 - v

In [10]:
save_results("hiv_rtv_50", fold_accuracies, fold_roc_aucs, fold_reports)
save_classification_report("hiv_rtv_50", avg_classification_report)

Results saved to reports/nosf/classification_results2_FNN.json
Classification report saved to reports/nosf/classification_reports2_ffnn.json


RES 75

In [44]:
# Load dataset
matrix_folder_antiinflam = 'data/matrices/mat_nosf/mat75/aip_antiinflam_matrix'
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.5118 - loss: 0.8414 - val_accuracy: 0.5941 - val_loss: 0.7448
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5493 - loss: 0.7684 - val_accuracy: 0.5941 - val_loss: 0.7309
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5892 - loss: 0.7400 - val_accuracy: 0.6000 - val_loss: 0.7193
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6242 - loss: 0.7050 - val_accuracy: 0.6118 - val_loss: 0.7104
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7143 - loss: 0.6501 - val_accuracy: 0.6294 - val_loss: 0.7105
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7561 - loss: 0.6131 - val_accuracy: 0.6471 - val_loss: 0.7286
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8273 - loss: 0.5277 - val_accuracy: 0.6294 - val_loss: 0.7632
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8503 - loss: 0.5014 - v

In [45]:
save_results("aip_antiinflam_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [46]:
save_classification_report("aip_antiinflam_75", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [47]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_nosf/mat75/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.5235 - loss: 0.8889 - val_accuracy: 0.8116 - val_loss: 0.7707
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5258 - loss: 0.7994 - val_accuracy: 0.7826 - val_loss: 0.7413
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6102 - loss: 0.7376 - val_accuracy: 0.7536 - val_loss: 0.7150
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6146 - loss: 0.7244 - val_accuracy: 0.8116 - val_loss: 0.6818
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7216 - loss: 0.6588 - val_accuracy: 0.7826 - val_loss: 0.6399
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7695 - loss: 0.6135 - val_accuracy: 0.7971 - val_loss: 0.5974
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8744 - loss: 0.5280 - val_accuracy: 0.8116 - val_loss: 0.5586
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9143 - loss: 0.4775 - v

In [48]:
save_results("amp_antibp_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [49]:
save_classification_report("amp_antibp_75", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [50]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_nosf/mat75/amp_antibp2_matrix'
label_file_antipb = 'data/labels/amp_antibp2.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.5096 - loss: 0.8634 - val_accuracy: 0.7688 - val_loss: 0.7505
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5257 - loss: 0.7810 - val_accuracy: 0.8125 - val_loss: 0.7092
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6522 - loss: 0.7161 - val_accuracy: 0.8000 - val_loss: 0.6583
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7199 - loss: 0.6521 - val_accuracy: 0.8000 - val_loss: 0.6081
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8299 - loss: 0.5360 - val_accuracy: 0.7688 - val_loss: 0.5967
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8747 - loss: 0.4805 - val_accuracy: 0.7625 - val_loss: 0.6146
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9195 - loss: 0.4217 - val_accuracy: 0.7125 - val_loss: 0.6707
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9430 - loss: 0.3715 - v

In [51]:
save_results("amp_antibp2_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [52]:
save_classification_report("amp_antibp2_75", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [53]:
# Load dataset
matrix_folder_csamp = 'data/matrices/mat_nosf/mat75/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 122ms/step - accuracy: 0.5615 - loss: 0.9724 - val_accuracy: 0.4286 - val_loss: 0.9294
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5003 - loss: 0.8799 - val_accuracy: 0.4286 - val_loss: 0.8601
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5163 - loss: 0.8859 - val_accuracy: 0.4286 - val_loss: 0.8167
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5273 - loss: 0.8483 - val_accuracy: 0.5238 - val_loss: 0.7926
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5642 - loss: 0.8049 - val_accuracy: 0.4762 - val_loss: 0.7808
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5242 - loss: 0.7900 - val_accuracy: 0.5238 - val_loss: 0.7739
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5296 - loss: 0.7965 - val_accuracy: 0.5714 - val_loss: 0.7688
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.6485 - loss: 0.7182 - val_accuracy: 0.

In [54]:
save_results("amp_csamp_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [55]:
save_classification_report("amp_csamp_75", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [56]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/mat_nosf/mat75/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.4819 - loss: 0.8870 - val_accuracy: 0.5600 - val_loss: 0.7881
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4749 - loss: 0.8483 - val_accuracy: 0.5200 - val_loss: 0.7629
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5108 - loss: 0.7902 - val_accuracy: 0.5600 - val_loss: 0.7554
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5704 - loss: 0.7375 - val_accuracy: 0.5400 - val_loss: 0.7487
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5761 - loss: 0.7453 - val_accuracy: 0.5600 - val_loss: 0.7423
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6184 - loss: 0.7149 - val_accuracy: 0.5600 - val_loss: 0.7377
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6457 - loss: 0.7187 - val_accuracy: 0.5800 - val_loss: 0.7328
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6295 - loss: 0.7001 - v

In [57]:
save_results("hiv_ddi_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [58]:
save_classification_report("hiv_ddi_75", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [59]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/mat_nosf/mat75/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.4682 - loss: 0.9456 - val_accuracy: 0.5000 - val_loss: 0.7979
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5005 - loss: 0.8249 - val_accuracy: 0.5000 - val_loss: 0.7649
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5159 - loss: 0.7899 - val_accuracy: 0.5000 - val_loss: 0.7588
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5495 - loss: 0.7691 - val_accuracy: 0.5000 - val_loss: 0.7509
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5639 - loss: 0.7646 - val_accuracy: 0.5000 - val_loss: 0.7409
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5287 - loss: 0.7903 - val_accuracy: 0.5000 - val_loss: 0.7330
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6201 - loss: 0.7293 - val_accuracy: 0.5250 - val_loss: 0.7258
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6709 - loss: 0.6803 - v

In [60]:
save_results("hiv_lpv_75", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [61]:
save_classification_report("hiv_lpv_75", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [11]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_nosf/mat75/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.5202 - loss: 0.9135 - val_accuracy: 0.5763 - val_loss: 0.7538
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5872 - loss: 0.7738 - val_accuracy: 0.5763 - val_loss: 0.7203
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5378 - loss: 0.7731 - val_accuracy: 0.7458 - val_loss: 0.6959
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5898 - loss: 0.7510 - val_accuracy: 0.7458 - val_loss: 0.6685
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6781 - loss: 0.6749 - val_accuracy: 0.7627 - val_loss: 0.6410
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7233 - loss: 0.6377 - val_accuracy: 0.7627 - val_loss: 0.6133
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7647 - loss: 0.5901 - val_accuracy: 0.7627 - val_loss: 0.5893
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7953 - loss: 0.5719 - v

In [12]:
save_results("hiv_rtv_75", fold_accuracies, fold_roc_aucs, fold_reports)
save_classification_report("hiv_rtv_75", avg_classification_report)

Results saved to reports/nosf/classification_results2_FNN.json
Classification report saved to reports/nosf/classification_reports2_ffnn.json


RES 100

In [62]:
# Load dataset
matrix_folder_antiinflam = 'data/matrices/mat_nosf/mat100/aip_antiinflam_matrix'  
label_file_antiinflam = 'data/labels/aip_antiinflam.txt' 
matrices, labels = load_data(matrix_folder_antiinflam, label_file_antiinflam)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.5424 - loss: 0.8372 - val_accuracy: 0.5941 - val_loss: 0.7453
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5856 - loss: 0.7539 - val_accuracy: 0.5941 - val_loss: 0.7347
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.6015 - loss: 0.7380 - val_accuracy: 0.5941 - val_loss: 0.7253
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.6294 - loss: 0.6967 - val_accuracy: 0.6059 - val_loss: 0.7175
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7187 - loss: 0.6341 - val_accuracy: 0.6118 - val_loss: 0.7277
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7927 - loss: 0.5591 - val_accuracy: 0.6176 - val_loss: 0.7566
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.8540 - loss: 0.4914 - val_accuracy: 0.6176 - val_loss: 0.8029
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8842 - loss: 0.4403 - v

In [63]:
save_results("aip_antiinflam_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [64]:
save_classification_report("aip_antiinflam_100", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [65]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_nosf/mat100/amp_antibp_matrix'
label_file_antipb = 'data/labels/amp_antibp.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.4968 - loss: 0.8864 - val_accuracy: 0.6232 - val_loss: 0.7634
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4933 - loss: 0.8299 - val_accuracy: 0.7246 - val_loss: 0.7452
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5550 - loss: 0.7519 - val_accuracy: 0.8261 - val_loss: 0.7234
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5679 - loss: 0.7406 - val_accuracy: 0.8551 - val_loss: 0.6926
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6893 - loss: 0.6770 - val_accuracy: 0.8261 - val_loss: 0.6506
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7701 - loss: 0.6308 - val_accuracy: 0.8261 - val_loss: 0.6083
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8601 - loss: 0.5176 - val_accuracy: 0.8551 - val_loss: 0.5709
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9204 - loss: 0.4625 - v

In [66]:
save_results("amp_antibp_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [67]:
save_classification_report("amp_antibp_100", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [68]:
# Load dataset
matrix_folder_antipb = 'data/matrices/mat_nosf/mat100/amp_antibp2_matrix'
label_file_antipb = 'data/labels/amp_antibp2.txt'
matrices, labels = load_data(matrix_folder_antipb, label_file_antipb)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.4992 - loss: 0.8677 - val_accuracy: 0.7750 - val_loss: 0.7464
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5315 - loss: 0.7654 - val_accuracy: 0.8562 - val_loss: 0.6910
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6848 - loss: 0.6763 - val_accuracy: 0.8000 - val_loss: 0.6344
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7744 - loss: 0.5945 - val_accuracy: 0.7937 - val_loss: 0.5940
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8929 - loss: 0.4756 - val_accuracy: 0.7563 - val_loss: 0.6043
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9454 - loss: 0.4000 - val_accuracy: 0.7688 - val_loss: 0.6230
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9521 - loss: 0.3480 - val_accuracy: 0.7500 - val_loss: 0.6747
Epoch 8/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9647 - loss: 0.3018 - v

In [69]:
save_results("amp_antibp2_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [70]:
save_classification_report("amp_antibp2_100", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [71]:
# Load dataset
matrix_folder_csamp = 'data/matrices/mat_nosf/mat100/amp_csamp_matrix'  
label_file_csamp = 'data/labels/amp_csamp.txt'  
matrices, labels = load_data(matrix_folder_csamp, label_file_csamp)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 319ms/step - accuracy: 0.4133 - loss: 0.9570 - val_accuracy: 0.5714 - val_loss: 0.8159
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.5205 - loss: 0.8753 - val_accuracy: 0.5714 - val_loss: 0.7799
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.5413 - loss: 0.7979 - val_accuracy: 0.5714 - val_loss: 0.7636
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.4901 - loss: 0.8130 - val_accuracy: 0.5714 - val_loss: 0.7571
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.5299 - loss: 0.7824 - val_accuracy: 0.5714 - val_loss: 0.7539
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.6497 - loss: 0.7245 - val_accuracy: 0.5714 - val_loss: 0.7499
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.5629 - loss: 0.7693 - val_accuracy: 0.5714 - val_loss: 0.7451
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.5990 - loss: 0.7226 - val_accuracy: 0.

In [72]:
save_results("amp_csamp_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [73]:
save_classification_report("amp_csamp_100", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [74]:
# Load dataset
matrix_folder_hivddi = 'data/matrices/mat_nosf/mat100/hiv_ddi_matrix' 
label_file_hivddi = 'data/labels/hiv_ddi.txt'  
matrices, labels = load_data(matrix_folder_hivddi, label_file_hivddi)
# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - accuracy: 0.4760 - loss: 0.8940 - val_accuracy: 0.4800 - val_loss: 0.7741
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5478 - loss: 0.7850 - val_accuracy: 0.4800 - val_loss: 0.7584
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5170 - loss: 0.7800 - val_accuracy: 0.5400 - val_loss: 0.7518
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5156 - loss: 0.7776 - val_accuracy: 0.5800 - val_loss: 0.7440
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6403 - loss: 0.7338 - val_accuracy: 0.6000 - val_loss: 0.7384
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6769 - loss: 0.6969 - val_accuracy: 0.6200 - val_loss: 0.7335
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.7032 - loss: 0.6650 - val_accuracy: 0.6000 - val_loss: 0.7302
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7410 - loss: 0.6570 - v

In [75]:
save_results("hiv_ddi_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [76]:
save_classification_report("hiv_ddi_100", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [77]:
# Load dataset
matrix_folder_hivlpv = 'data/matrices/mat_nosf/mat100/hiv_lpv_matrix'  
label_file_hivlpv = 'data/labels/hiv_lpv.txt' 
matrices, labels = load_data(matrix_folder_hivlpv, label_file_hivlpv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.5022 - loss: 1.0337 - val_accuracy: 0.5000 - val_loss: 0.8535
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.4854 - loss: 0.9240 - val_accuracy: 0.5000 - val_loss: 0.7808
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.4935 - loss: 0.8152 - val_accuracy: 0.5000 - val_loss: 0.7551
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5715 - loss: 0.7733 - val_accuracy: 0.7250 - val_loss: 0.7458
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5865 - loss: 0.7551 - val_accuracy: 0.5500 - val_loss: 0.7417
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5362 - loss: 0.7538 - val_accuracy: 0.5000 - val_loss: 0.7370
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5139 - loss: 0.7770 - val_accuracy: 0.5250 - val_loss: 0.7314
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5713 - loss: 0.7597 - v

In [78]:
save_results("hiv_lpv_100", fold_accuracies, fold_roc_aucs, fold_reports)

Results saved to reports/nosf/classification_results2_FNN.json


In [79]:
save_classification_report("hiv_lpv_100", avg_classification_report)

Classification report saved to reports/nosf/classification_reports2_ffnn.json


In [13]:
# Load dataset
matrix_folder_hivrtv = 'data/matrices/mat_nosf/mat100/hiv_rtv_matrix'  
label_file_hivrtv = 'data/labels/hiv_rtv.txt' 
matrices, labels = load_data(matrix_folder_hivrtv, label_file_hivrtv)

# Preprocess data
X = matrices.reshape(matrices.shape[0], -1)  # Input features
y = labels  # Target labels

# Perform 10-fold cross-validation
fold_accuracies, fold_roc_aucs, fold_reports, fold_confusion_matrices, avg_classification_report = cross_validate_model(X, y, n_splits=10)



Training Fold 1/10...
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 110ms/step - accuracy: 0.5128 - loss: 0.8787 - val_accuracy: 0.5763 - val_loss: 0.7374
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.5229 - loss: 0.7889 - val_accuracy: 0.7797 - val_loss: 0.7050
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.6077 - loss: 0.7336 - val_accuracy: 0.7627 - val_loss: 0.6692
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6665 - loss: 0.6867 - val_accuracy: 0.7797 - val_loss: 0.6302
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.7680 - loss: 0.5992 - val_accuracy: 0.7797 - val_loss: 0.5994
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.8224 - loss: 0.5627 - val_accuracy: 0.7797 - val_loss: 0.5742
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.8209 - loss: 0.5314 - val_accuracy: 0.7797 - val_loss: 0.5534
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.8200 - loss: 0.5206 - 

In [14]:
save_results("hiv_rtv_100", fold_accuracies, fold_roc_aucs, fold_reports)
save_classification_report("hiv_rtv_100", avg_classification_report)

Results saved to reports/nosf/classification_results2_FNN.json
Classification report saved to reports/nosf/classification_reports2_ffnn.json
